# Prompt Engineering & In-Context Learning

**Companion lesson:** https://ml-viz.vercel.app/courses/building-with-llms/01-prompt-engineering

A from-scratch, runnable implementation of the concepts in the lesson — pure NumPy, no API keys required.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import re

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## Decoding from a next-token distribution

An LLM emits **logits** over the vocabulary; decoding turns them into text. We implement temperature scaling and top-p (nucleus) sampling on a fixed toy distribution.

In [ ]:
TOKENS = ['mat','floor','sofa','roof','table','bed','grass','moon']
LOGITS = np.array([3.2, 2.6, 2.1, 1.0, 1.7, 1.3, 0.4, -0.5])

def softmax_t(logits, T):
    z = logits / max(T, 1e-3)
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()

for T in [0.5, 1.0, 1.5]:
    p = softmax_t(LOGITS, T)
    print(f'T={T}: ' + '  '.join(f'{t}={pi:.2f}' for t, pi in zip(TOKENS, p)))

Low temperature concentrates mass on the top token (greedy); high temperature flattens it. Let's visualise.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
for ax, T in zip(axes, [0.5, 1.0, 1.5]):
    p = softmax_t(LOGITS, T)
    ax.bar(TOKENS, p, color='#6366f1')
    ax.set_title(f'temperature = {T}')
    ax.tick_params(axis='x', rotation=45)
axes[0].set_ylabel('probability')
plt.tight_layout(); plt.show()

## Top-p (nucleus) sampling

Keep the smallest set of tokens whose cumulative probability reaches `p`, drop the tail, renormalise.

In [ ]:
def nucleus(probs, p):
    order = np.argsort(probs)[::-1]
    cum = np.cumsum(probs[order])
    cutoff = np.searchsorted(cum, p) + 1  # how many to keep
    keep = order[:cutoff]
    out = np.zeros_like(probs)
    out[keep] = probs[keep] / probs[keep].sum()
    return out

p = softmax_t(LOGITS, 1.0)
q = nucleus(p, 0.9)
print('kept tokens:', [TOKENS[i] for i in np.where(q > 0)[0]])
print('renormalised:', np.round(q[q > 0], 3))

## ✏️ Your turn

Implement **top-k** sampling: keep only the `k` highest-probability tokens, then renormalise.

In [ ]:
def top_k(probs, k):
    # TODO(you): zero out all but the k largest probabilities, then renormalise.
    out = np.zeros_like(probs)
    # ...
    return out

res = top_k(softmax_t(LOGITS, 1.0), 3)
assert np.count_nonzero(res) == 3
assert abs(res.sum() - 1.0) < 1e-9
print('passed ✓')

<details><summary>Solution</summary>

```python
def top_k(probs, k):
    idx = np.argsort(probs)[::-1][:k]
    out = np.zeros_like(probs)
    out[idx] = probs[idx] / probs[idx].sum()
    return out
```

</details>